In [0]:
%sql

USE CATALOG medalhao;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "medalhao"
silver_db = "silver"
gold_db = "gold"

In [0]:
# Carregamento de todas as tabelas limpas da camada Silver
df_info = spark.table(f"{catalogo}.{silver_db}.tb_info_filmes")
df_fin = spark.table(f"{catalogo}.{silver_db}.tb_financeiro_filmes")
df_metrics = spark.table(f"{catalogo}.{silver_db}.tb_metricas_engajamento")
df_reviews = spark.table(f"{catalogo}.{silver_db}.tb_avaliacoes_usuarios")
df_generos = spark.table(f"{catalogo}.{silver_db}.tb_generos")
df_pessoas_empresas = spark.table(f"{catalogo}.{silver_db}.tb_pessoas_empresas")

print("Todas as tabelas da Silver foram carregadas com SUCESSO para o ambiente da Gold!")

Todas as tabelas da Silver foram carregadas com SUCESSO para o ambiente da Gold!


### Criação das Dimensões de Filmes, Gêneros, Pessoas e Produtoras (dim_movies, dim_genres, dim_people, dim_companies)
Extrai catálogos únicos e deduplicados para Filmes, Gêneros, Pessoas (filtrando estritamente por Ator, Diretor e Roteirista) e Produtoras, gerando suas respectivas Surrogate Keys.

In [0]:
# 1. Dimensão de Filmes
janela_movie = Window.orderBy("id_filme")
df_dim_movies = (
    df_info
    .select("id_filme", "titulo", "data_lancamento", "ano_lancamento", "duracao_minutos", "idioma_original", "status_filme", "sinopse")
    .dropDuplicates(["id_filme"])
    .withColumn("sk_movie_id", F.row_number().over(janela_movie).cast("bigint"))
    .select("sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento", "duracao_minutos", "idioma_original", "status_filme", "sinopse")
)
df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.dim_movies")

# 2. Dimensão de Gêneros (Compreende todos os 19 gêneros, incluindo TV Movie se presente na silver)
janela_genre = Window.orderBy("nome_genero")
df_dim_genres = (
    df_generos
    .select(F.col("genero").alias("nome_genero"))
    .filter(F.col("nome_genero").isNotNull() & (F.col("nome_genero") != ""))
    .dropDuplicates(["nome_genero"])
    .withColumn("sk_genre_id", F.row_number().over(janela_genre).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)
df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.dim_genres")

# 3. Dimensão de Pessoas (Ator, Diretor, Roteirista)
janela_person = Window.orderBy("nome_pessoa", "tipo_pessoa")
df_dim_people = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .filter(F.col("nome_pessoa").isNotNull() & (F.col("nome_pessoa") != ""))
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"])
    .withColumn("sk_person_id", F.row_number().over(janela_person).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)
df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.dim_people")

# 4. Dimensão de Produtoras
janela_company = Window.orderBy("nome_produtora")
df_dim_companies = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .filter(F.col("nome_produtora").isNotNull() & (F.col("nome_produtora") != ""))
    .dropDuplicates(["nome_produtora"])
    .withColumn("sk_company_id", F.row_number().over(janela_company).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)
df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.dim_companies")

print("✓ Dimensões criadas e gravadas com sucesso!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ Dimensões criadas e gravadas com sucesso!


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### Criação da Dimensão de Avaliações (dim_reviews) e Tabelas Bridge
Agrega as avaliações de usuários por filme, calculando a contagem total e a nota média arredondada, mapeando a chave estrangeira para o respectivo filme.
Estabelece os relacionamentos N:N entre os filmes e suas dimensões multivaloradas.

In [0]:
df_movies_sk = spark.sql(f"SELECT sk_movie_id, id_filme FROM {catalogo}.{gold_db}.dim_movies")
df_genres_sk = spark.sql(f"SELECT sk_genre_id, nome_genero FROM {catalogo}.{gold_db}.dim_genres")
df_people_sk = spark.sql(f"SELECT sk_person_id, nome_pessoa, tipo_pessoa FROM {catalogo}.{gold_db}.dim_people")
df_companies_sk = spark.sql(f"SELECT sk_company_id, nome_produtora FROM {catalogo}.{gold_db}.dim_companies")

# Dimensão de Reviews Agregada
df_reviews_agregado = (
    df_reviews
    .groupBy("id_filme")
    .agg(
        F.count("nota_usuario").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
)
janela_review = Window.orderBy("sk_movie_id")
df_dim_reviews = (
    df_reviews_agregado
    .join(df_movies_sk, on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(janela_review).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.dim_reviews")

# Bridge Filme x Gênero
df_bridge_genre = (
    df_generos
    .select("id_filme", F.col("genero").alias("nome_genero"))
    .join(df_movies_sk, on="id_filme", how="inner")
    .join(df_genres_sk, on="nome_genero", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates(["sk_movie_id", "sk_genre_id"])
)
df_bridge_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.bridge_movie_genre")

# Bridge Filme x Pessoa
df_bridge_person = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select("id_filme", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .join(df_movies_sk, on="id_filme", how="inner")
    .join(df_people_sk, on=["nome_pessoa", "tipo_pessoa"], how="inner")
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates(["sk_movie_id", "sk_person_id"])
)
df_bridge_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.bridge_movie_person")

# Bridge Filme x Produtora
df_bridge_company = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("id_filme", F.col("nome_entidade").alias("nome_produtora"))
    .join(df_movies_sk, on="id_filme", how="inner")
    .join(df_companies_sk, on="nome_produtora", how="inner")
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates(["sk_movie_id", "sk_company_id"])
)
df_bridge_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db}.bridge_movie_company")

print("Dimensão de reviews e tabelas Bridge geradas com sucesso!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dimensão de reviews e tabelas Bridge geradas com sucesso!


### Criação da Tabela Fato (fact_movies_performance)
Centraliza as métricas financeiras (orçamento, receita e lucro em USD e BRL) e indicadores de engajamento por filme, associando à dimensão principal via Surrogate Key.

In [0]:
# A Fato consolida métricas estritamente de filmes lançados
df_dim_movies_lancados = spark.sql(f"SELECT sk_movie_id, id_filme FROM {catalogo}.{gold_db}.dim_movies WHERE status_filme = 'Lançado'")

df_fact_movies_performance = (
    df_dim_movies_lancados
    .join(df_fin, on="id_filme", how="left")
    .join(df_metrics, on="id_filme", how="left")
    .select(
        F.col("sk_movie_id"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
)

(
    df_fact_movies_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_db}.fact_movies_performance")
)

print("gold.fact_movies_performance gerada e persistida com sucesso!")

gold.fact_movies_performance gerada e persistida com sucesso!


### Geração da Tabela de Contexto GenAI
Monta o documento aplicando tratamentos defensivos contra nulos (coalesce) e conversões de tipo

In [0]:
df_dim_movies_all = spark.sql(f"SELECT sk_movie_id, id_filme, titulo, ano_lancamento, sinopse FROM {catalogo}.{gold_db}.dim_movies")
df_fact_perf = spark.table(f"{catalogo}.{gold_db}.fact_movies_performance")
df_bridge_person_all = spark.table(f"{catalogo}.{gold_db}.bridge_movie_person")
df_dim_people_all = spark.table(f"{catalogo}.{gold_db}.dim_people")

# Junta atores e diretores em listas de string únicas por filme para usar no texto do RAG
df_pessoas_filme = (
    df_bridge_person_all
    .join(df_dim_people_all, "sk_person_id", "inner")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(", ", F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Ator", F.col("nome_pessoa"))))).alias("atores"),
        F.concat_ws(", ", F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Diretor", F.col("nome_pessoa"))))).alias("diretores")
    )
)

# Montagem do template em frase corrida. O coalesce garante que campos nulos não quebrem a string inteira
df_genai = (
    df_dim_movies_all
    .join(df_fact_perf, "sk_movie_id", "left")
    .join(df_pessoas_filme, "sk_movie_id", "left")
    .select(
        F.col("id_filme").alias("movie_id"),
        F.col("titulo").alias("title"),
        F.concat(
            F.lit("O filme "),
            F.coalesce(F.col("titulo"), F.lit("Título não informado")),
            F.lit(", lançado no ano de "),
            F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado")),
            F.lit(", faturou "),
            F.coalesce(F.concat(F.lit("US$ "), F.col("receita_usd").cast("string")), F.lit("valor não informado")),
            F.lit(" e teve um custo de "),
            F.coalesce(F.concat(F.lit("US$ "), F.col("orcamento_usd").cast("string")), F.lit("valor não informado")),
            F.lit(". Estrelado por "),
            F.when(F.col("atores").isNull() | (F.col("atores") == ""), F.lit("elenco não informado")).otherwise(F.col("atores")),
            F.lit(" e dirigido por "),
            F.when(F.col("diretores").isNull() | (F.col("diretores") == ""), F.lit("diretor não informado")).otherwise(F.col("diretores")),
            F.lit(", o filme possui a seguinte sinopse: "),
            F.coalesce(F.col("sinopse"), F.lit("sinopse não informada")),
            F.lit(".")
        ).alias("llm_context_document")
    )
)

(
    df_genai.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_db}.gold_genai_movies_context")
)

print("gold.gold_genai_movies_context gerada com sucesso!")

gold.gold_genai_movies_context gerada com sucesso!


### Desafio de Analytics
Consultas analíticas para responder às 6 perguntas de negócio utilizando as tabelas da camada Gold.

In [0]:
fact = spark.table(f"{catalogo}.{gold_db}.fact_movies_performance")
movies = spark.table(f"{catalogo}.{gold_db}.dim_movies")

# Pega a data mais recente da base para servir de teto nos recortes temporais
data_limite = (
    movies
    .filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date()))
    .agg(F.max("data_lancamento"))
    .first()[0]
)
print(f"Data limite para recortes temporais: {data_limite}")

# 1. Receita total convertida em Reais somando a base toda
display(fact.agg(F.sum("receita_brl").alias("receita_total_brl")))

# 2. Top 5 filmes mais populares
display(
    fact.join(movies.select("sk_movie_id", "titulo"), "sk_movie_id")
    .filter(F.col("popularidade").isNotNull())
    .select("titulo", "popularidade")
    .orderBy(F.desc("popularidade"))
    .limit(5)
)

# 3. Volume de filmes agrupados por cada gênero
display(
    spark.table(f"{catalogo}.{gold_db}.bridge_movie_genre")
    .join(spark.table(f"{catalogo}.{gold_db}.dim_genres"), "sk_genre_id")
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.desc("qtd_filmes"))
)

# 4. Os 10 maiores bilheterias em dólar ordenadas com RANK()
janela_receita = Window.orderBy(F.desc("receita_usd"))
display(
    fact.join(movies.select("sk_movie_id", "titulo"), "sk_movie_id")
    .filter(F.col("receita_usd").isNotNull())
    .withColumn("posicao", F.rank().over(janela_receita))
    .select("posicao", "titulo", "receita_usd", "receita_brl")
    .orderBy("posicao")
    .limit(10)
)

# 5. Ator com mais filmes lançados nos últimos 2 anos a partir da data limite
atores_2_anos = (
    spark.table(f"{catalogo}.{gold_db}.bridge_movie_person")
    .join(spark.table(f"{catalogo}.{gold_db}.dim_people").filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id")
    .join(movies.select("sk_movie_id", "data_lancamento", "status_filme"), "sk_movie_id")
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") <= F.lit(data_limite)) &
        (F.col("data_lancamento") >= F.add_months(F.lit(data_limite), -24))
    )
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes"))
)
janela_atores = Window.orderBy(F.desc("qtd_participacoes"))
display(
    atores_2_anos.withColumn("posicao", F.rank().over(janela_atores))
    .filter(F.col("posicao") == 1)
    .select("nome_pessoa", "qtd_participacoes")
)

# 6. Produtora que mais lucrou (em USD) nos últimos 5 anos
produtoras_5_anos = (
    spark.table(f"{catalogo}.{gold_db}.bridge_movie_company")
    .join(spark.table(f"{catalogo}.{gold_db}.dim_companies"), "sk_company_id")
    .join(movies.select("sk_movie_id", "data_lancamento", "status_filme"), "sk_movie_id")
    .join(fact.select("sk_movie_id", "lucro_usd"), "sk_movie_id")
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") <= F.lit(data_limite)) &
        (F.col("data_lancamento") >= F.add_months(F.lit(data_limite), -60)) &
        F.col("lucro_usd").isNotNull()
    )
    .groupBy("nome_produtora")
    .agg(F.sum("lucro_usd").alias("lucro_total_usd"))
)
janela_produtoras = Window.orderBy(F.desc("lucro_total_usd"))
display(
    produtoras_5_anos.withColumn("posicao", F.rank().over(janela_produtoras))
    .filter(F.col("posicao") == 1)
    .select("nome_produtora", "lucro_total_usd")
)

Data limite para recortes temporais: 2026-02-19


receita_total_brl
834731384018.25


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0


nome_genero,qtd_filmes
Drama,32127
Documentary,18928
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
Tv Movie,4066


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14439320000.00
2,Avatar: The Way of Water,2320250281.00,11965298674.09
3,AVENGERS: INFINITY WAR,2052415039.00,10584099114.62
4,spider-man: no way home,1921847111.00,9910773366.72
5,The Lion King,1663075401.00,8576313535.42
6,Top Gun: Maverick,1488732821.00,7677246284.61
7,Barbie,1428545028.00,7366863854.89
8,The Super Mario Bros. Movie,1355725263.00,6991339608.76
9,Black Panther,1349926083.00,6961433817.42
10,Star Wars: The Last Jedi,1332698830.00,6872594596.43


nome_pessoa,qtd_participacoes
Kevin Hart,64


nome_produtora,lucro_total_usd
Universal Pictures,5772329679.00
